# 🦀 Day 1: SQLite with `rusqlite`
---

Today we enter the world of databases! We'll use **rusqlite** — safe Rust bindings for SQLite.

- Creating and opening databases
- Creating tables and inserting data
- Querying with parameterized statements
- Mapping rows to structs
- Transactions

In [ ]:
:dep rusqlite = { version = "0.31", features = ["bundled"] }

In [ ]:
use rusqlite::{Connection, params, Result};

// Open an in-memory database (no file needed!)
let conn = Connection::open_in_memory().unwrap();

// Create a table
conn.execute(
    "CREATE TABLE users (
        id    INTEGER PRIMARY KEY AUTOINCREMENT,
        name  TEXT NOT NULL,
        email TEXT NOT NULL UNIQUE,
        age   INTEGER
    )",
    [],
).unwrap();

println!("✅ Table 'users' created!");

In [ ]:
// INSERT with parameterized queries (prevents SQL injection)
conn.execute(
    "INSERT INTO users (name, email, age) VALUES (?1, ?2, ?3)",
    params!["Alice", "alice@example.com", 30],
).unwrap();

conn.execute(
    "INSERT INTO users (name, email, age) VALUES (?1, ?2, ?3)",
    params!["Bob", "bob@example.com", 25],
).unwrap();

conn.execute(
    "INSERT INTO users (name, email, age) VALUES (?1, ?2, ?3)",
    params!["Charlie", "charlie@example.com", 35],
).unwrap();

println!("✅ Inserted 3 users");

In [ ]:
// Query a single row
let user_name: String = conn.query_row(
    "SELECT name FROM users WHERE id = ?1",
    params![1],
    |row| row.get(0),
).unwrap();
println!("User #1: {}", user_name);

// Query multiple rows with prepare + query_map
#[derive(Debug)]
struct User {
    id: i32,
    name: String,
    email: String,
    age: Option<i32>,
}

let mut stmt = conn.prepare("SELECT id, name, email, age FROM users").unwrap();
let users: Vec<User> = stmt.query_map([], |row| {
    Ok(User {
        id: row.get(0)?,
        name: row.get(1)?,
        email: row.get(2)?,
        age: row.get(3)?,
    })
}).unwrap()
  .filter_map(|r| r.ok())
  .collect();

println!("\nAll users:");
for u in &users {
    println!("  {:?}", u);
}

In [ ]:
// UPDATE and DELETE
let rows_updated = conn.execute(
    "UPDATE users SET age = ?1 WHERE name = ?2",
    params![31, "Alice"],
).unwrap();
println!("Updated {} row(s)", rows_updated);

let rows_deleted = conn.execute(
    "DELETE FROM users WHERE name = ?1",
    params!["Charlie"],
).unwrap();
println!("Deleted {} row(s)", rows_deleted);

// Verify
let count: i32 = conn.query_row("SELECT COUNT(*) FROM users", [], |row| row.get(0)).unwrap();
println!("Remaining users: {}", count);

In [ ]:
// Transactions — all-or-nothing operations
let tx = conn.unchecked_transaction().unwrap();

tx.execute(
    "INSERT INTO users (name, email, age) VALUES (?1, ?2, ?3)",
    params!["Diana", "diana@example.com", 28],
).unwrap();

tx.execute(
    "INSERT INTO users (name, email, age) VALUES (?1, ?2, ?3)",
    params!["Eve", "eve@example.com", 22],
).unwrap();

tx.commit().unwrap();
println!("✅ Transaction committed!");

// Verify all users
let mut stmt = conn.prepare("SELECT id, name, email, age FROM users ORDER BY id").unwrap();
let users: Vec<String> = stmt.query_map([], |row| {
    let name: String = row.get(1)?;
    let email: String = row.get(2)?;
    Ok(format!("{} <{}>", name, email))
}).unwrap()
  .filter_map(|r| r.ok())
  .collect();

println!("All users after transaction:");
for u in &users {
    println!("  {}", u);
}

In [ ]:
// Error handling — what happens with constraint violations?
let result = conn.execute(
    "INSERT INTO users (name, email, age) VALUES (?1, ?2, ?3)",
    params!["Alice2", "alice@example.com", 99],  // duplicate email!
);

match result {
    Ok(_) => println!("Inserted (unexpected)"),
    Err(e) => println!("Expected error: {}", e),
}

// query_row returns Err when no rows found
let result = conn.query_row(
    "SELECT name FROM users WHERE id = ?1",
    params![999],
    |row| row.get::<_, String>(0),
);
println!("Missing row: {:?}", result);

## 🏋️ Exercise

Create a `tasks` table with columns: `id`, `title`, `completed` (boolean as INTEGER),
and `priority` (TEXT: "low", "medium", "high"). Implement full CRUD operations.

In [ ]:
// YOUR CODE HERE
// 1. Create the tasks table
// 2. Insert 3-4 tasks with different priorities
// 3. Query all incomplete tasks
// 4. Mark a task as completed
// 5. Delete completed tasks
// 6. Print remaining tasks

## 🎯 Key Takeaways

1. **rusqlite** provides safe, ergonomic SQLite access in Rust
2. Always use **parameterized queries** (`?1`, `params![]`) to prevent SQL injection
3. `query_row()` for single results, `prepare()` + `query_map()` for multiple rows
4. **Transactions** ensure atomic operations — all succeed or all roll back
5. SQLite's `:memory:` mode is perfect for testing and prototyping

---
**Tomorrow:** ORMs — abstracting database access with the Repository pattern! 🏗️